# 实验三 · Stream 与任务级并行 —— 传输与计算的重叠

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐ 基础　|　**预计时长**：40 分钟

异步接口本身不缩短单次传输的耗时，它只是让主机线程在传输期间可以做别的事。这段空出来的时间要怎么用起来，是本实验要回答的问题。

做法是把搬进、计算、搬出这三段切块后交错下发到多条 Stream 上，让一块数据的计算与另一块数据的传输同时进行。这个思路学生并不陌生：第三章用寄存器轮转让访存与运算重叠，第六章用 `TQue` 与 Double Buffer 让 GM↔UB 的搬运与 Vector 计算重叠。本实验是同一个模式在系统尺度上的第三次出现。

> **实验说明**
> 1. 本实验的核心内容有四点：Stream 的任务队列语义、默认 Stream 与显式 Stream 之间不存在隐式同步、三段流水线的重叠机制，以及流水线加速比的上界由什么决定。
> 2. 本实验采用**四个版本的递进设计**：单流整块同步、单流分块异步、双流分块异步、多流分块异步，用以分离**分块**与**多流**各自的作用。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 计算段用 `aclnnAdd` 作为占位算子，此处只把它当作一个会在设备上占用时间的任务使用，不展开两段式接口本身。
> 6. 本实验的耗时一律用**主机侧墙钟时间**测量，不使用设备侧计时。
> 7. 本实验直接使用锁页内存、异步接口与半带宽规模 $S_{1/2}$，默认读者已经掌握这三者。**§12 需要填入本机实测的 $S_{1/2}$**，填法见该节代码注释。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说出 Stream 的三条语义：同一 Stream 内保序、硬件资源充足时流间并行、相对主机线程异步
- 指出默认 Stream 与显式创建的 Stream 之间**不存在隐式同步**
- 正确选用 `aclrtSynchronizeDevice`、`aclrtSynchronizeStream` 与 `aclrtStreamQuery` 三种同步粒度
- 解释为什么把数据切块并改用异步接口在单条 Stream 上不产生任何重叠，**重叠的前提是多条队列**
- 用 $T_{\text{pipe}} = \sum_j t_j + (k-1)\, t_{\max}$ 预测流水线耗时，并由它算出加速比的上界
- 由本机的流数扫描判断加速比在第几条 Stream 上到顶，并说明到顶的原因是某类硬件资源被占满


## 🗺️ 学习路径

1. **准备阶段**：理解异步下发之后主机线程不再被阻塞，这段时间可以用于组织任务级并行
2. **概念建立**：Stream 的任务队列语义，同一队列内保序、跨队列之间才可能并行
3. **接口辨析**：三种同步粒度各自等待的是哪些任务
4. **模型建立**：把三段流水线的耗时写成公式，由它得到加速比的上界与到达上界的条件
5. **对照设计**：由整块与分块、单流与多流组合出四个版本，使重叠的来源可以逐项分离
6. **观测重叠**：由版本对比与阶段分解，判断收益究竟来自哪一步
7. **结果分析**：区分重叠的前提、上界的含义，以及实测达不到上界的三类原因


## 1. 背景与动机：异步之后空出来的时间要怎么用

有一条已知的结论：`aclrtMemcpyAsync` 在锁页内存上真正异步，但异步并不降低单次传输的耗时，反而多出了约一份固定开销。若在每次异步下发之后立刻调用 `aclrtSynchronizeStream`，让出的空隙没有被利用，因此只测到了代价、没有测到收益。

本实验补上后半句。设一段计算需要经过三步：

1. 把输入从主机搬到设备（H2D）；
2. 在设备上计算；
3. 把输出从设备搬回主机（D2H）。

若整块数据一次搬完、一次算完、一次搬回，则三步严格串行，总耗时是三段之和。但这三步分别由不同的硬件单元完成：H2D 与 D2H 由 DMA 引擎负责，计算由 AI Core 负责。**三个单元在串行执行时，任一时刻都只有一个在工作，另外两个空闲。**

把数据切成 $k$ 块之后，情况可以改变：第 2 块的 H2D 可以与第 1 块的计算同时进行，第 1 块的 D2H 又可以与第 2 块的计算同时进行。这就是流水线。


### 1.1 三级流水在三个尺度上的重演

本实验要建立的机制，在本课程中已经出现过两次。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 尺度 | 出处 | 搬入 | 计算 | 搬出 | 重叠机制 | 一次搬运量 |
| --- | --- | --- | --- | --- | --- | --- |
| 寄存器级 | 第三章 ARM NEON | `vld1q_f32` | `vmlaq_f32` | `vst1q_f32` | 寄存器轮转与编译器软流水 | 16 B |
| 片上级 | 第六章 Ascend C | `DataCopy` GM→UB | `Add` / `Mul` | `DataCopy` UB→GM | `TQue` 与 Double Buffer | KB 级 |
| 系统级 | **第七章 本实验** | `aclrtMemcpyAsync` H2D | 算子任务 | `aclrtMemcpyAsync` D2H | **多 Stream** | MB 级 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">尺度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">出处</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">搬入</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">搬出</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">重叠机制</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">一次搬运量</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">寄存器级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第三章 ARM NEON</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>vld1q_f32</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>vmlaq_f32</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>vst1q_f32</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">寄存器轮转与编译器软流水</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16 B</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第六章 Ascend C</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DataCopy</code> GM→UB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Add</code> / <code>Mul</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>DataCopy</code> UB→GM</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TQue</code> 与 Double Buffer</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">KB 级</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">系统级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>第七章 本实验</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpyAsync</code> H2D</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子任务</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpyAsync</code> D2H</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多 Stream</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">MB 级</td>
</tr>
</tbody>
</table>

三个尺度上是同一个模式：**两端搬运、中间计算，用多重缓冲把搬运藏进计算里。** 差别只在缓冲区放在哪里（寄存器 / UB / HBM）、由谁调度（编译器 / TPipe / Runtime）、跨越什么介质（片内总线 / 片上网络 / 主机与设备之间的总线）。

双流重叠把三段之和压到最长的一段，与第六章 Double Buffer 把搬运藏进计算是同一个机制，只是尺度不同。

### 1.2 本实验不测什么

1. **不测设备侧耗时。** 本实验全部用主机侧墙钟时间。三段各自的耗时由下发一段、同步、停止计时的方式分别测得，这种测法把主机侧的下发与同步开销一并计入，本实验不把它们分开。
2. **不做跨流依赖。** 本实验的每一块数据在同一条 Stream 上完成三个阶段，块与块之间没有依赖，因此不需要 Event。


## 2. Stream 的语义

### 2.1 三条基本规则

Stream 的定义是：**Stream 描述了一个在 Host 下发并在 Device 上执行的任务队列。** 由此展开三条规则：

1. **同一 Stream 内保序**：任务按进入队列的顺序依次执行。
2. **流间可并行，但不保证并行**：当硬件资源充足时，不同 Stream 上的任务会被调度到不同的硬件资源上并行执行；**当硬件资源不足时，不同 Stream 上的任务可能串行执行。**
3. **相对主机线程异步**：主机线程下发任务后立即返回，需要用同步接口等待完成。

下面两张图用来把可并行与可能串行这两句话落到具体的任务序列上，本实验随后用实测把它们量化。

<img src="images/07.03_stream_parallel.png" alt="两条 Stream 上任务的下发顺序与执行时间线" width="769px">

图中主机侧按顺序下发三个核函数，Kernel1 与 Kernel3 进入 stream1、Kernel2 进入 stream2。执行时间线上，stream1 的两个任务按下发顺序先后执行（规则一），而 stream2 的任务与它们并行（规则二）。**下发顺序与执行顺序不是一回事**——这正是本实验要利用的性质。

第二条规则的后半句是本实验后半段的关键：

> 当硬件资源不足时，不同 Stream 上的任务可能串行执行。

流数增加到一定程度后加速比不再提升，原因就在这里。任务能否并行，最终由设备上的任务调度器根据可用的执行单元决定。

<img src="images/07.03_task_scheduling.png" alt="任务从 Stream 队列到设备执行单元的调度过程" height="520">

上一张图给出的是下发顺序与执行时间线的对照，这一张给出的是中间发生了什么：主机侧的下发只是把任务加入队列（步骤 1），真正决定何时、在哪个执行单元上运行的是设备侧的**任务调度器**（步骤 2）；执行完成后由步骤 5、6 把结果回填给主机。

调度器可以分派的执行单元不止图中的两类。下图列出全部：

<img src="images/07.03_hardware_accelerators.png" alt="Runtime 协同调度的多种硬件加速器" width="700px">

其中 **DMA 承接数据拷贝任务与内存 Cache 任务、AI Core 承接矢量与矩阵计算任务**，两者是不同的执行单元——这是本实验三个阶段能够重叠的物理基础；而同类任务共享同一组单元，这是重叠有上限的物理原因——§5.1 的上界公式建立在这张图上。需要说明的是，不同代 AI 处理器支持的硬件加速器不同，须以实际硬件用户手册为准。


### 2.2 默认 Stream：一条必须记住的规则

调用 `aclrtSetDevice` 或 `aclrtCreateContext` 时，Runtime 会自动创建一个默认 Stream，每个 Context 拥有一个。使用它的方式是在需要 Stream 参数的接口中传入 `nullptr`。

值得注意的是：

> Runtime 中的 Stream 均为非阻塞式 Stream，**默认 Stream 不会跟显式创建的 Stream 进行隐式同步**。

这条规则值得强调，是因为它与部分其它异构编程模型的传统语义相反。在那些模型中，默认流具有全局同步的性质：向默认流下发的任务会等待所有其它流上的任务完成，其它流上的任务也会等待默认流。**AscendCL 没有这种性质**——默认 Stream 与显式 Stream 是完全对等的，彼此之间不存在任何隐式的先后关系。

因此在 AscendCL 中，「用默认 Stream 做一次同步以确保前面的任务都完成」这个做法是无效的。要等待其它 Stream，必须显式调用同步接口。§13 会用计时的方式把这条规则验证一遍。

默认 Stream 的其余规则：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 规则 | 说明 |
| --- | --- |
| 创建时机 | `aclrtSetDevice` 与 `aclrtCreateContext` 各隐式创建一个 |
| 数量关系 | 每个 Context 一个；**同一 Context 的不同主机线程共享同一个默认 Stream** |
| 使用方式 | 需要 Stream 入参的接口传 `nullptr`；没有 Stream 入参的接口（如 `aclrtMemcpy`）**不使用**默认 Stream |
| 销毁 | 不能调用 `aclrtDestroyStream` 显式销毁；`aclrtResetDevice` 时自动销毁 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">规则</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">创建时机</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSetDevice</code> 与 <code>aclrtCreateContext</code> 各隐式创建一个</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数量关系</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 Context 一个；<strong>同一 Context 的不同主机线程共享同一个默认 Stream</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使用方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要 Stream 入参的接口传 <code>nullptr</code>；没有 Stream 入参的接口（如 <code>aclrtMemcpy</code>）<strong>不使用</strong>默认 Stream</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">销毁</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不能调用 <code>aclrtDestroyStream</code> 显式销毁；<code>aclrtResetDevice</code> 时自动销毁</td>
</tr>
</tbody>
</table>

选用建议：

> 默认 Context、默认 Stream 一般适用于简单应用，用户仅需要一个 Device 的计算场景下。多线程应用程序建议使用显式创建的 Context 和 Stream。

本实验全部使用显式创建的 Stream，仅在 §13 为验证语义而用到默认 Stream。


### 2.3 创建、销毁与数量上限

```cpp
aclError aclrtCreateStream(aclrtStream *stream);
aclError aclrtDestroyStream(aclrtStream stream);
```

`aclrtCreateStream` 不接受优先级参数，需要指定优先级时应改用下面的 `aclrtCreateStreamWithConfig`。**销毁时若 Stream 上仍有未完成的任务，接口会等待任务完成后再销毁**——这一点意味着 `aclrtDestroyStream` 自带同步语义，但依赖它来做同步是不好的写法，因为它把等待与释放两件事混在了一起。

单个 Device 上可创建的 Stream 数量存在上限，该上限包含默认 Stream 与 Runtime 内部用于同步的 Stream，因此实际可显式创建的数量要减去已存在的部分。`aclrtGetStreamAvailableNum` 可以查询当前剩余可用的数量，§4 的自检会打印它——本实验只用到 8 条，不会触及。

若需要在创建时指定属性，可改用：

```cpp
aclError aclrtCreateStreamWithConfig(aclrtStream *stream, uint32_t priority,
                                     uint32_t flag);
```

可以通过该接口设置优先级，并：

> Stream 的优先级在 Device 范围内生效，而不是在 Context 范围内生效。

§4 的探针会打印 `aclrtDeviceGetStreamPriorityRange` 的结果，可以据此了解本机报告的范围。

`flag` 参数的两个取值：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 取值 | 作用 | 代价 |
| --- | --- | --- |
| `ACL_STREAM_FAST_LAUNCH` | 创建 Stream 时预申请内部资源，**下发任务更快** | 创建更慢，内存占用增加 |
| `ACL_STREAM_FAST_SYNC` | `aclrtSynchronizeStream` 改为主动轮询，任务一完成立即返回 | 增加 CPU 开销 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">取值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">作用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">代价</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_STREAM_FAST_LAUNCH</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">创建 Stream 时预申请内部资源，<strong>下发任务更快</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">创建更慢，内存占用增加</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_STREAM_FAST_SYNC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeStream</code> 改为主动轮询，任务一完成立即返回</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">增加 CPU 开销</td>
</tr>
</tbody>
</table>

这两个 flag 按其命名针对的正是本实验要测的两项开销——下发耗时与同步延迟，动手练习第 3 题会实测它们在本机上的实际效果。


## 3. 显式同步

### 3.1 三种同步粒度

异步接口调用成功只表示任务下发成功，不表示执行完成。等待完成有三种粒度：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 等待范围 | 是否阻塞主机线程 | 典型用途 |
| --- | --- | --- | --- |
| `aclrtSynchronizeDevice()` | **当前 Context 下全部 Stream**（含默认 Stream） | 阻塞 | 程序收尾；确认所有工作完成 |
| `aclrtSynchronizeStream(stream)` | 指定的一条 Stream | 阻塞 | 取回该 Stream 的计算结果 |
| `aclrtStreamQuery(stream, &status)` | 指定的一条 Stream | **不阻塞** | 轮询进度；判断资源可否复用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">等待范围</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否阻塞主机线程</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">典型用途</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeDevice()</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>当前 Context 下全部 Stream</strong>（含默认 Stream）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">阻塞</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">程序收尾；确认所有工作完成</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeStream(stream)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指定的一条 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">阻塞</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">取回该 Stream 的计算结果</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtStreamQuery(stream, &amp;status)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指定的一条 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不阻塞</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">轮询进度；判断资源可否复用</td>
</tr>
</tbody>
</table>

`aclrtStreamQuery` 的输出是一个枚举：

```cpp
typedef enum aclrtStreamStatus {
  ACL_STREAM_STATUS_COMPLETE = 0,   // Stream 上的所有任务已完成
  ACL_STREAM_STATUS_NOT_READY = 1,  // Stream 上至少有一个任务未完成
  ACL_STREAM_STATUS_RESERVED = 0xFFFF,
} aclrtStreamStatus;
```

它是本实验用来**在不改变程序行为的前提下观察 Stream 状态**的手段：调用同步接口会改变时序，而查询不会。§13 依靠它来验证默认 Stream 的语义。

本实验的流水线在下发完全部任务之后，对每一条 Stream 依次调用 `aclrtSynchronizeStream`。也可以只调用一次 `aclrtSynchronizeDevice` 达到同样的效果，两者的差别在于前者能分辨是哪一条 Stream 最后完成。


## 4. 环境准备与检查

先把 CANN 的环境变量导入 Jupyter 进程，并创建代码目录。


In [ ]:
!mkdir -p src_stream

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


本实验的自检有三项与 Stream 直接相关：算子库（计算段要调用 `aclnnAdd`）、**当前 Device 上剩余可用的 Stream 数**，以及硬件支持的 Stream 优先级范围。

In [ ]:
import os, shutil, subprocess

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]


# 按优先级在库目录中查找库，返回第一个存在的库名（不含 lib 前缀与 .so 后缀）
def find_lib(candidates):
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


print("=" * 62)
print(" 一、CANN 与编译器")
print("=" * 62)
print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
print("g++              :", shutil.which("g++") or "⚠️  未找到")

requirements = [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("算子公共数据类型", ["nnopbase"]),
    ("Math 类算子", ["opapi_math"]),
]
selected = []
for purpose, candidates in requirements:
    name = find_lib(candidates)
    print(f"{purpose:<18} 候选 {candidates} -> {name}")
    if name is not None:
        selected.append(name)

ACL_LIBDIRS = ["-L" + path for path in lib_dirs]
ACL_LIBS = ["-l" + name for name in selected]
# 只包含 acl/acl.h 的程序仅需链接 Runtime 库，单独保存一份供探针程序使用
ACL_RT_LIB = ["-l" + selected[0]] if selected else []
print("库目录           :", " ".join(ACL_LIBDIRS))
print("链接参数         :", " ".join(ACL_LIBS))

ready = bool(ascend_home) and shutil.which("g++") is not None and len(selected) == 3
print()
print("✅ 环境就绪，可以开始实验。" if ready else "⚠️  环境不完整，请检查上面的输出。")


下面这个单元格查询与 Stream 有关的两项设备信息。它写成一个独立的小程序，是因为这两项都必须在 `aclrtSetDevice` 之后才能查询。


In [ ]:
%%writefile src_stream/stream_probe.cpp
// Prints the stream-related device limits used by this lab: how many streams
// are still available, and the priority range the hardware reports.
#include <cstdint>
#include <cstdio>

#include "acl/acl.h"

int main() {
  if (aclInit(nullptr) != ACL_SUCCESS || aclrtSetDevice(0) != ACL_SUCCESS) {
    std::printf("[ERR] failed to initialise the device\n");
    return 1;
  }

  uint32_t available = 0;
  if (aclrtGetStreamAvailableNum(&available) == ACL_SUCCESS) {
    std::printf("[PROBE] streams_available=%u\n", available);
  }

  int32_t least = 0;
  int32_t greatest = 0;
  if (aclrtDeviceGetStreamPriorityRange(&least, &greatest) == ACL_SUCCESS) {
    std::printf("[PROBE] priority_least=%d priority_greatest=%d\n", least,
                greatest);
  }

  aclrtStream default_stream = nullptr;
  if (aclrtCtxGetCurrentDefaultStream(&default_stream) == ACL_SUCCESS) {
    std::printf("[PROBE] default_stream_is_null=%d\n",
                default_stream == nullptr ? 1 : 0);
  }

  aclrtResetDevice(0);
  aclFinalize();
  return 0;
}


这个探针只包含 `acl/acl.h`，因此只链接 Runtime 库。


In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_stream/stream_probe.cpp", "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_RT_LIB
    + ["-o", "src_stream/stream_probe"]
)
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print((proc.stdout + proc.stderr).strip())
else:
    print(
        subprocess.run(
            ["./src_stream/stream_probe"], capture_output=True, text=True
        ).stdout
    )


探针的三行输出各自对应一件事，都值得读一读。

**`streams_available`** 是当前 Device 上剩余可用的 Stream 数。它与 §2.3 提到的硬件上限之间的差值，就是此刻已经存在的 Stream 数量——其中包含本进程的默认 Stream 与 Runtime 内部用于同步的 Stream。请确认这个数远大于本实验要用的 8 条；只要如此，§11 的流数扫描就不会受它限制。

**`priority_least` 与 `priority_greatest`** 是硬件报告的优先级范围。请注意两点：其一，**数值较小的一端是较高的优先级**，因此 `greatest` 通常是较小的那个数；其二，**这个范围不是退化的单点区间，但这并不意味着优先级一定生效**——能否生效要查本机产品的接口文档，查询接口返回了范围并不构成证据。

**`default_stream_is_null`** 表明 `aclrtCtxGetCurrentDefaultStream` 返回的是默认 Stream 的实际句柄，它**不是** `nullptr`。在接口调用时传 `nullptr` 表示使用默认 Stream，这是一种约定的写法，而不是说默认 Stream 的句柄就是空指针。


## 5. 本实验的性能模型

### 5.1 三段流水线

设一次完整的工作被切成 $k$ 块，每块都要依次经过 $s$ 个阶段。本实验中 $s = 3$：H2D、计算、D2H。记第 $j$ 个阶段处理一块所需的时间为 $t_j$。

**串行执行**时每块依次走完三段，总耗时为

$$T_{\text{serial}} = k \sum_{j=1}^{s} t_j$$

**理想流水**时，各阶段由不同的硬件单元承担，可以同时工作。第一块数据仍要完整地走过全部 $s$ 个阶段，此后每多一块只多占用一个节拍，而节拍由最慢的那一段决定：

$$T_{\text{pipe}} = \sum_{j=1}^{s} t_j + (k - 1)\, t_{\max}, \qquad t_{\max} = \max_j t_j$$

常见的写法 $T_{\text{pipe}} = (k + s - 1)\, t_{\max}$ 是上式在**各段耗时相等**时的特例：此时 $\sum_j t_j = s\, t_{\max}$，代入即得。本实验的三段并不相等，因此必须用上面的一般形式，否则在分块数很小时会算出比串行还慢的荒谬结果。

两者相除得到加速比：

$$\text{Speedup} = \frac{k \sum_j t_j}{\sum_j t_j + (k - 1)\, t_{\max}}$$

它在 $k = 1$ 时等于 1（一块数据无从重叠），随 $k$ 单调上升，并且

$$\text{Speedup} \xrightarrow{\;k \to \infty\;} \frac{\sum_j t_j}{t_{\max}}$$

这个极限就是本实验的**上界**：三段之和除以其中最长的一段。它只有在三段完全相等时才等于 3；只要有一段明显长于另两段，它就会成为节拍，其余阶段在等它。$S_\infty$ 由本机 §9 打印的三段耗时直接算出，是一个可以立刻验算的数。

需要说明的是，这个公式默认三段各占一种互不相干的硬件资源。**H2D 与 D2H 走的是同一条主机与设备之间的链路，它们能否真的同时进行，取决于本机的 DMA 通路是单向复用还是双向独立。** 因此实测加速比达不到 $S_\infty$ 是正常的，§12 ③ 会给出判断的办法。

### 5.2 测量要点

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 要点 | 本实验的做法 | 不这样做的后果 |
| --- | --- | --- |
| **首次触碰** | 分配主机缓冲区后立即写满全部字节 | 首次传输同时付出缺页中断的代价 |
| **预热** | 每个配置先跑 2 次不计时 | 首次运行包含一次性的初始化开销 |
| **多次重复取平均** | 每个配置重复若干次取平均 | 单次测量的抖动可达数十个百分点 |
| **分别记录下发与总耗时** | 下发循环结束时记一次表，全部同步后再记一次 | 无法判断瓶颈在主机还是在设备 |
| **同步全部 Stream 后再停计时** | 逐条调用 `aclrtSynchronizeStream` | 只同步一条会漏掉其它 Stream 上未完成的任务 |
| **张量与内存只准备一次** | 在计时循环之外创建张量、申请内存 | 把准备开销混进流水线耗时 |
| **基线的取法** | 串行基线按整块测三段，流水版本按每块 $1/k$ 测 | 由整块基线算出的预测值不是流水版本的严格上界，实测可以超过它 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验的做法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">不这样做的后果</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>首次触碰</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分配主机缓冲区后立即写满全部字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首次传输同时付出缺页中断的代价</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>预热</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个配置先跑 2 次不计时</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首次运行包含一次性的初始化开销</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多次重复取平均</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个配置重复若干次取平均</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单次测量的抖动可达数十个百分点</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>分别记录下发与总耗时</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">下发循环结束时记一次表，全部同步后再记一次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无法判断瓶颈在主机还是在设备</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>同步全部 Stream 后再停计时</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">逐条调用 <code>aclrtSynchronizeStream</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只同步一条会漏掉其它 Stream 上未完成的任务</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>张量与内存只准备一次</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">在计时循环之外创建张量、申请内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把准备开销混进流水线耗时</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>基线的取法</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">串行基线按整块测三段，流水版本按每块 $1/k$ 测</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由整块基线算出的预测值不是流水版本的严格上界，实测可以超过它</td>
</tr>
</tbody>
</table>

第四条与第五条是本实验特有的。第五条尤其容易出错：多流场景下只同步最后一条 Stream 会得到一个明显偏小的耗时，那不是流水线快，而是有一部分任务根本没有等它完成。

第七条要单独说明。§9 与其后各节打印的模型预测值都由**整块**串行基线的三段分解算出，而流水版本每块只有 $1/k$ 的规模，每次算子调用的规模也随之变小——而传输效率随规模变化。因此这个预测值**不是严格意义上的上界**，实测有可能超过它。


## 6. 版本设计总览

四个版本沿一条递进的路线排列，每一步只改变一件事。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 分块 | 复制接口 | Stream 数 | 这一步改变了什么 | 预期 |
| --- | --- | --- | --- | --- | --- |
| **v1** | 整块 | 同步 `aclrtMemcpy` | 1 | 基线 | 三段严格串行，总耗时为三段之和 |
| **v2** | $k$ 块 | 异步 `aclrtMemcpyAsync` | 1 | 分块 + 换成异步接口 | **几乎无改善** |
| **v3** | $k$ 块 | 异步 | 2 | 把块轮流分配到两条 Stream | 出现重叠，加速比明显大于 1 |
| **v4** | $k$ 块 | 异步 | $S$ | 继续增加 Stream 数 | 收益递减，最终不再提升 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">分块</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">复制接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Stream 数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">这一步改变了什么</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">预期</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">整块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同步 <code>aclrtMemcpy</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基线</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三段严格串行，总耗时为三段之和</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$k$ 块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步 <code>aclrtMemcpyAsync</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分块 + 换成异步接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>几乎无改善</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$k$ 块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把块轮流分配到两条 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">出现重叠，加速比明显大于 1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$k$ 块</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">继续增加 Stream 数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">收益递减，最终不再提升</td>
</tr>
</tbody>
</table>

**v2 是本实验的关键对照。** 它把数据切成了 $k$ 块，也换用了异步接口，两项改动看上去都指向重叠，但结果是几乎没有改善。原因在 §2.1 的第一条规则：**同一条 Stream 内的任务按进入队列的顺序依次执行**。第 2 块的 H2D 排在第 1 块的 D2H 后面，必须等它完成，因此三段依然是串起来跑的，只是被切成了 $3k$ 段而不是 3 段。

这个道理学生在第六章见过：同一条 `TQue` 上的 `EnQue` 与 `DeQue` 也是保序的，要让搬运与计算重叠，靠的不是把数据切碎，而是**用两块缓冲区轮转**。本实验的 Stream 就是那两块缓冲区在系统尺度上的对应物。

v2 相对 v1 还会略慢一些：分块之后每块都要付一次传输固定开销 $\alpha$，计算也从 1 次调用变成 $k$ 次调用。这两项额外开销在 v1 中是不存在的。

### 6.1 计算任务的选择

三个阶段中的计算段用 `aclnnAdd` 充当，计算内容是 $\text{out} = \text{in} + 1.0 \times \text{bias}$，其中 `bias` 常驻设备侧、不参与每块的传输。选它的理由是**本实验关心的是任务在设备上占用多长时间，而不是它算的是什么**：`aclnnAdd` 接口简单、耗时可控，正合用。

把同一次计算重复 $R$ 次并不改变结果（每次都是同样的输入算出同样的输出），因此校验与 $R$ 无关。这是有意的设计：让配平手段不干扰正确性检查。

### 6.2 与官方建议的另一种分流方式

《应用开发指南》「编写高性能应用程序的建议」小节给出的另一种组织方式是**按算子执行硬件划分 Stream**：

> AI 处理器中包含多种硬件加速器，例如 AI Core、AI CPU、DVPP、Random 等，这些硬件加速器对应不同类型的任务，建议多 Stream 的创建按照算子执行硬件划分。

本实验采用的是另一种：每条 Stream 承担一整条 H2D→计算→D2H 流水，块与块之间轮转。两者的差别在于依赖关系——按硬件划分时，第 $i$ 块的计算与它的 H2D 落在不同 Stream 上，必须用跨流同步机制显式表达这层依赖，本实验不采用它，正是为了把不需要跨流依赖这个前提保持住（见 §1.2 第二条）。

同一节还把**单线程中创建并使用多个 Stream**列为首选推荐方式，本实验正是单线程多 Stream。


## 7. 程序实现

程序分五段写入 `src_stream/acl_stream.cpp`。第一段为覆盖写，其余四段为追加写，因此**必须按顺序执行**；中途修改了某一段，需要从 7.1 开始重新执行。

### 7.1 头文件、常量与错误检查

程序接受五个命令行参数：模式、总数据量（MB）、分块数、流数、计算重复次数。五种模式分别对应 §10 到 §13 的五组测量。


`ACL_CHECK` 是本章统一的错误检查写法：失败时打印接口名、返回码与错误消息，然后立即返回。


In [ ]:
%%writefile src_stream/acl_stream.cpp
/**
 * Parallel Computing, Chapter 7, Lab 3: Streams and Task-Level Parallelism
 *
 * This program measures how much of the host-to-device transfer, the device
 * computation and the device-to-host transfer can be overlapped by splitting
 * the work into chunks and distributing the chunks over several streams.
 *
 * Four versions form a ladder. v1 is the serial baseline: one chunk and
 * synchronous copies. v2 splits the work into k chunks and switches to the
 * asynchronous copy API but keeps a single stream. v3 and v4 spread the same
 * chunks over two and over S streams respectively.
 *
 * Usage: acl_stream <mode> <total_mb> <chunks> <streams> <repeat>
 *   mode = all | streams | chunks | repeat | default
 */
#include <cstdint>  // int32_t, int64_t, uint64_t
#include <cstdio>   // std::printf, std::fprintf
#include <cstdlib>  // std::atoi
#include <cstring>  // std::strcmp
#include <ctime>    // clock_gettime, timespec

#include "acl/acl.h"            // Runtime resource management APIs
#include "aclnnop/aclnn_add.h"  // Single-operator API of Add

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kMaxStreams = 8;
constexpr int kMaxChunks = 256;
constexpr int kWarmupRuns = 2;
constexpr float kAlphaValue = 1.0f;
constexpr float kBiasValue = 1.0f;
constexpr double kRelTolerance = 1.0e-6;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 7.2 计时工具与算子封装

`SubmitAdd` 把算子的两段式调用包成一次提交。需要注意的是**第一段接口是在主机侧执行的**：它做入参校验、输出 Shape 推导与 Tiling，返回一个执行器与所需的 workspace 大小；只有第二段接口才把任务下发到 Stream。程序单独记录的 `submit_ms`，主要就是这一段的耗时。


In [ ]:
%%writefile -a src_stream/acl_stream.cpp

// Returns a monotonic timestamp in milliseconds. CLOCK_MONOTONIC increases
// steadily since system start and is unaffected by wall-clock adjustments.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Submits one Add task to a stream. Built-in CANN operators use a two-phase
// interface. The first phase runs on the host: it validates the arguments,
// infers the output shape, performs tiling and reports the workspace size.
// Only the second phase submits the kernel to the stream, so the cost of the
// first phase is part of the host-side submission time measured by this lab.
int SubmitAdd(const aclTensor* self, const aclTensor* other,
              const aclScalar* alpha, aclTensor* out, void* workspace,
              uint64_t workspace_capacity, aclrtStream stream) {
  uint64_t needed = 0;
  aclOpExecutor* executor = nullptr;
  ACL_CHECK(
      aclnnAddGetWorkspaceSize(self, other, alpha, out, &needed, &executor));
  if (needed > workspace_capacity) {
    // Terminating path: the executor returned above is not consumed by a
    // second-phase call. The program exits right after this, so nothing is
    // leaked for long; on any path that keeps running, the executor must be
    // handed to aclnnAdd exactly once (see ConfigureChunks).
    std::fprintf(stderr,
                 "[ERR] api=SubmitAdd code=- msg=workspace %llu exceeds the "
                 "reserved %llu bytes\n",
                 static_cast<unsigned long long>(needed),
                 static_cast<unsigned long long>(workspace_capacity));
    return ACL_ERROR_INVALID_PARAM;
  }
  ACL_CHECK(aclnnAdd(workspace, needed, executor, stream));
  return ACL_SUCCESS;
}


### 7.3 资源集合

一个 `Pipeline` 结构持有全部资源。主机侧的两块缓冲区都用 `aclrtMallocHost` 申请：**只有锁页内存上的 `aclrtMemcpyAsync` 才是真正异步的**，用可分页内存会让本实验的全部版本退化成同一个。

每条 Stream 需要一块独立的 workspace。同一条 Stream 上的任务顺序执行，可以共用一块；不同 Stream 上的任务可能同时执行，共用会导致数据竞争。

`ConfigureChunks` 按当前分块数重建张量。张量在计时循环之外创建，避免把准备开销混进测量。


In [ ]:
%%writefile -a src_stream/acl_stream.cpp

// Holds every resource the pipeline needs. The host buffers are page-locked
// because aclrtMemcpyAsync is only truly asynchronous on page-locked memory.
struct Pipeline {
  int64_t total_elems;
  int64_t chunk_elems;
  int chunks;
  int streams;
  float* host_in;
  float* host_out;
  void* dev_in;
  void* dev_out;
  void* dev_bias;
  void* dev_probe;  // 4-byte scratch buffer used only by the §13 probe
  aclrtStream stream[kMaxStreams];
  void* workspace[kMaxStreams];
  uint64_t workspace_size;
  aclTensor* in_tensor[kMaxChunks];
  aclTensor* out_tensor[kMaxChunks];
  aclTensor* bias_tensor;
  aclScalar* alpha;
  int64_t shape[1];
  int64_t stride[1];
};

// Creates a one-dimensional float tensor over `elems` elements at `addr`. The
// view description and the storage description coincide because every chunk
// is a contiguous slice of a larger buffer.
aclTensor* MakeTensor(Pipeline* p, void* addr) {
  return aclCreateTensor(p->shape, 1, ACL_FLOAT, p->stride, 0, ACL_FORMAT_ND,
                         p->shape, 1, addr);
}

int DestroyTensors(Pipeline* p) {
  for (int c = 0; c < p->chunks; ++c) {
    if (p->in_tensor[c] != nullptr) {
      ACL_CHECK(aclDestroyTensor(p->in_tensor[c]));
      p->in_tensor[c] = nullptr;
    }
    if (p->out_tensor[c] != nullptr) {
      ACL_CHECK(aclDestroyTensor(p->out_tensor[c]));
      p->out_tensor[c] = nullptr;
    }
  }
  if (p->bias_tensor != nullptr) {
    ACL_CHECK(aclDestroyTensor(p->bias_tensor));
    p->bias_tensor = nullptr;
  }
  return ACL_SUCCESS;
}

// Rebuilds the tensors for a given chunk count and reserves one workspace per
// stream. Called outside every timed region.
int ConfigureChunks(Pipeline* p, int chunks) {
  // The sweep tables are edited by the exercises, so the same two
  // invariants main checks are re-checked here, where chunk_elems is
  // actually computed.
  if (chunks <= 0 || chunks > kMaxChunks || p->total_elems % chunks != 0) {
    std::fprintf(stderr,
                 "[ERR] api=ConfigureChunks code=- msg=chunk count %d is "
                 "out of range 1..%d or does not divide %lld elements\n",
                 chunks, kMaxChunks,
                 static_cast<long long>(p->total_elems));
    return ACL_ERROR_INVALID_PARAM;
  }
  ACL_CHECK(DestroyTensors(p));
  p->chunks = chunks;
  p->chunk_elems = p->total_elems / chunks;
  p->shape[0] = p->chunk_elems;

  for (int c = 0; c < chunks; ++c) {
    const int64_t offset = static_cast<int64_t>(c) * p->chunk_elems;
    p->in_tensor[c] = MakeTensor(p, static_cast<float*>(p->dev_in) + offset);
    p->out_tensor[c] = MakeTensor(p, static_cast<float*>(p->dev_out) + offset);
  }
  p->bias_tensor = MakeTensor(p, p->dev_bias);

  uint64_t needed = 0;
  aclOpExecutor* executor = nullptr;
  ACL_CHECK(aclnnAddGetWorkspaceSize(p->in_tensor[0], p->bias_tensor, p->alpha,
                                     p->out_tensor[0], &needed, &executor));
  if (needed > p->workspace_size) {
    std::fprintf(stderr,
                 "[ERR] api=ConfigureChunks code=- msg=the operator asks for "
                 "%llu bytes of workspace, more than the reserved amount\n",
                 static_cast<unsigned long long>(needed));
    return ACL_ERROR_INVALID_PARAM;
  }
  // The executor returned by the first phase is consumed by the second phase,
  // so it is submitted once here rather than left unused.
  ACL_CHECK(aclnnAdd(p->workspace[0], needed, executor, p->stream[0]));
  ACL_CHECK(aclrtSynchronizeStream(p->stream[0]));
  std::printf("[CHUNK] chunks=%d chunk_kb=%lld workspace_bytes=%llu\n", chunks,
              static_cast<long long>(p->chunk_elems * 4 / 1024),
              static_cast<unsigned long long>(needed));
  return ACL_SUCCESS;
}


`AllocPipeline` 按最大配置一次性申请全部内存。`kWorkspaceReserve` 预留的是算子 workspace 的上限——第一段接口返回的实际大小在 `ConfigureChunks` 中核对，超出即报错，而不是默默截断。

首次触碰在这里完成：主机缓冲区分配后立即写满，把缺页中断的代价挡在计时之外。


In [ ]:
%%writefile -a src_stream/acl_stream.cpp

namespace {
// Upper bound reserved for the operator workspace of one stream.
constexpr uint64_t kWorkspaceReserve = 16ULL << 20;
}  // namespace

int AllocPipeline(Pipeline* p, int64_t total_elems, int streams) {
  p->total_elems = total_elems;
  p->streams = streams;
  p->chunks = 0;
  p->workspace_size = kWorkspaceReserve;
  p->stride[0] = 1;
  const size_t bytes = static_cast<size_t>(total_elems) * sizeof(float);

  ACL_CHECK(aclrtMallocHost(reinterpret_cast<void**>(&p->host_in), bytes));
  ACL_CHECK(aclrtMallocHost(reinterpret_cast<void**>(&p->host_out), bytes));
  ACL_CHECK(aclrtMalloc(&p->dev_in, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc(&p->dev_out, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc(&p->dev_bias, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  // A dedicated scratch buffer for the probe task of §13. Writing to any
  // buffer the explicit stream is still reading would be a genuine data
  // race, because the default stream does not synchronize with it.
  ACL_CHECK(
      aclrtMalloc(&p->dev_probe, sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST));

  // First touch: writing every element forces the operating system to back
  // the page-locked buffers before any measurement starts.
  for (int64_t i = 0; i < total_elems; ++i) {
    p->host_in[i] = static_cast<float>(i % 1024) * 0.25f;
    p->host_out[i] = 0.0f;
  }
  // The bias is uploaded once and stays resident on the device, so it is not
  // part of the per-chunk transfer.
  for (int64_t i = 0; i < total_elems; ++i) {
    p->host_out[i] = kBiasValue;
  }
  ACL_CHECK(aclrtMemcpy(p->dev_bias, bytes, p->host_out, bytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  for (int64_t i = 0; i < total_elems; ++i) {
    p->host_out[i] = 0.0f;
  }

  for (int s = 0; s < streams; ++s) {
    ACL_CHECK(aclrtCreateStream(&p->stream[s]));
    ACL_CHECK(aclrtMalloc(&p->workspace[s], kWorkspaceReserve,
                          ACL_MEM_MALLOC_HUGE_FIRST));
  }
  float alpha_value = kAlphaValue;
  p->alpha = aclCreateScalar(&alpha_value, ACL_FLOAT);
  return ACL_SUCCESS;
}

int FreePipeline(Pipeline* p) {
  ACL_CHECK(DestroyTensors(p));
  if (p->alpha != nullptr) {
    ACL_CHECK(aclDestroyScalar(p->alpha));
  }
  for (int s = 0; s < p->streams; ++s) {
    ACL_CHECK(aclrtFree(p->workspace[s]));
    ACL_CHECK(aclrtDestroyStream(p->stream[s]));
  }
  ACL_CHECK(aclrtFree(p->dev_probe));
  ACL_CHECK(aclrtFree(p->dev_bias));
  ACL_CHECK(aclrtFree(p->dev_out));
  ACL_CHECK(aclrtFree(p->dev_in));
  ACL_CHECK(aclrtFreeHost(p->host_out));
  ACL_CHECK(aclrtFreeHost(p->host_in));
  return ACL_SUCCESS;
}

// The expected result of every version is out = in + alpha * bias, regardless
// of how many times the computation was repeated, because each repetition
// recomputes the same value from the same input. A relative-error tolerance
// is the right tool here: unlike a memory copy, this is arithmetic.
int VerifyResult(const Pipeline* p, const char* ver) {
  double max_rel = 0.0;
  for (int64_t i = 0; i < p->total_elems; ++i) {
    const double want =
        static_cast<double>(p->host_in[i]) +
        static_cast<double>(kAlphaValue) * static_cast<double>(kBiasValue);
    const double got = static_cast<double>(p->host_out[i]);
    const double denom = (want == 0.0) ? 1.0 : (want < 0.0 ? -want : want);
    const double err = (got - want < 0.0) ? (want - got) : (got - want);
    if (err / denom > max_rel) {
      max_rel = err / denom;
    }
  }
  const bool ok = max_rel <= kRelTolerance;
  std::printf("[VERIFY] ver=%s max_rel_err=%.3e result=%s\n", ver, max_rel,
              ok ? "PASS" : "FAIL");
  return ok ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}


### 7.4 四个版本的执行函数

`RunSerial` 是 v1。它把三段分别计时：每一段下发之后立即同步，因此三个时间之和就是总耗时。这三个数是 §5 全部预测的输入。

`RunPipelined` 是 v2 至 v4 的共同实现——三者的差别只在 `streams` 这一个参数上。**下发循环结束时记一次时钟得到 $T_{\text{submit}}$，全部 Stream 同步完成后再记一次时钟得到总耗时**，两者之差就是主机下发完成后设备仍在工作的时间。


In [ ]:
%%writefile -a src_stream/acl_stream.cpp

struct StageTimes {
  double h2d_ms;
  double comp_ms;
  double d2h_ms;
};

// v1: the serial baseline. The whole buffer is transferred in one piece with
// the synchronous copy API, and each of the three stages is timed separately.
int RunSerial(Pipeline* p, int repeat, StageTimes* out) {
  const size_t bytes = static_cast<size_t>(p->total_elems) * sizeof(float);
  const double t0 = GetTimeMs();
  ACL_CHECK(aclrtMemcpy(p->dev_in, bytes, p->host_in, bytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  const double t1 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) {
    ACL_CHECK(SubmitAdd(p->in_tensor[0], p->bias_tensor, p->alpha,
                        p->out_tensor[0], p->workspace[0], p->workspace_size,
                        p->stream[0]));
  }
  ACL_CHECK(aclrtSynchronizeStream(p->stream[0]));

  const double t2 = GetTimeMs();
  ACL_CHECK(aclrtMemcpy(p->host_out, bytes, p->dev_out, bytes,
                        ACL_MEMCPY_DEVICE_TO_HOST));
  const double t3 = GetTimeMs();

  out->h2d_ms = t1 - t0;
  out->comp_ms = t2 - t1;
  out->d2h_ms = t3 - t2;
  return ACL_SUCCESS;
}

// v2, v3 and v4: the chunks are submitted round-robin over `streams` streams.
// Every chunk goes through all three stages on the stream it was assigned to,
// so chunks on different streams can overlap while chunks on the same stream
// cannot.
int RunPipelined(Pipeline* p, int streams, int repeat, double* submit_ms,
                 double* total_ms) {
  const size_t chunk_bytes =
      static_cast<size_t>(p->chunk_elems) * sizeof(float);
  const double t0 = GetTimeMs();
  for (int c = 0; c < p->chunks; ++c) {
    const int slot = c % streams;
    aclrtStream stream = p->stream[slot];
    const int64_t offset = static_cast<int64_t>(c) * p->chunk_elems;
    ACL_CHECK(aclrtMemcpyAsync(static_cast<float*>(p->dev_in) + offset,
                               chunk_bytes, p->host_in + offset, chunk_bytes,
                               ACL_MEMCPY_HOST_TO_DEVICE, stream));
    for (int r = 0; r < repeat; ++r) {
      ACL_CHECK(SubmitAdd(p->in_tensor[c], p->bias_tensor, p->alpha,
                          p->out_tensor[c], p->workspace[slot],
                          p->workspace_size, stream));
    }
    ACL_CHECK(aclrtMemcpyAsync(p->host_out + offset, chunk_bytes,
                               static_cast<float*>(p->dev_out) + offset,
                               chunk_bytes, ACL_MEMCPY_DEVICE_TO_HOST, stream));
  }
  const double t1 = GetTimeMs();

  // Every stream must be synchronized. Waiting on only one of them would
  // report a total time that simply excludes the unfinished work.
  for (int s = 0; s < streams; ++s) {
    ACL_CHECK(aclrtSynchronizeStream(p->stream[s]));
  }
  const double t2 = GetTimeMs();

  *submit_ms = t1 - t0;
  *total_ms = t2 - t0;
  return ACL_SUCCESS;
}


### 7.5 测量循环、两种模式与主程序

`MeasurePipelined` 负责预热与重复取平均，并打印一条 `[PERF]` 记录行。两种模式共用它，靠 `tag` 字段区分：`all` 跑四个版本的对照，`streams` 跑流数扫描。


In [ ]:
%%writefile -a src_stream/acl_stream.cpp

int MeasurePipelined(Pipeline* p, const char* tag, const char* ver, int streams,
                     int repeat, int runs) {
  double submit_ms = 0.0;
  double total_ms = 0.0;
  for (int i = 0; i < kWarmupRuns; ++i) {
    ACL_CHECK(RunPipelined(p, streams, repeat, &submit_ms, &total_ms));
  }
  double submit_sum = 0.0;
  double total_sum = 0.0;
  for (int i = 0; i < runs; ++i) {
    ACL_CHECK(RunPipelined(p, streams, repeat, &submit_ms, &total_ms));
    submit_sum += submit_ms;
    total_sum += total_ms;
  }
  std::printf(
      "[PERF] tag=%s ver=%s chunks=%d streams=%d repeat=%d submit_ms=%.4f "
      "total_ms=%.4f\n",
      tag, ver, p->chunks, streams, repeat, submit_sum / runs,
      total_sum / runs);
  return ACL_SUCCESS;
}

// Measures the serial baseline and prints both its stage decomposition and a
// [PERF] line, so that every mode carries its own baseline.
int MeasureSerial(Pipeline* p, const char* tag, int repeat, int runs) {
  StageTimes st = {0.0, 0.0, 0.0};
  for (int i = 0; i < kWarmupRuns; ++i) {
    ACL_CHECK(RunSerial(p, repeat, &st));
  }
  StageTimes sum = {0.0, 0.0, 0.0};
  for (int i = 0; i < runs; ++i) {
    ACL_CHECK(RunSerial(p, repeat, &st));
    sum.h2d_ms += st.h2d_ms;
    sum.comp_ms += st.comp_ms;
    sum.d2h_ms += st.d2h_ms;
  }
  const double h2d = sum.h2d_ms / runs;
  const double comp = sum.comp_ms / runs;
  const double d2h = sum.d2h_ms / runs;
  std::printf(
      "[STAGE] tag=%s repeat=%d h2d_ms=%.4f comp_ms=%.4f d2h_ms=%.4f "
      "serial_ms=%.4f\n",
      tag, repeat, h2d, comp, d2h, h2d + comp + d2h);
  // v1 uses synchronous copies, so it has no separate submission phase.
  // submit_ms is reported as -1 and the notebook prints it as "-".
  std::printf(
      "[PERF] tag=%s ver=v1 chunks=1 streams=1 repeat=%d submit_ms=%.4f "
      "total_ms=%.4f\n",
      tag, repeat, -1.0, h2d + comp + d2h);
  return ACL_SUCCESS;
}

// Verifies that the default stream does not implicitly synchronize with an
// explicitly created stream. The probe task writes p->dev_probe, a buffer no
// other stream touches: writing to a buffer the explicit stream is still
// reading would be a real data race, precisely because there is no implicit
// synchronization. Only timings and stream states are observed.

namespace {

constexpr int kStreamSweep[] = {1, 2, 4, 6, 8};

constexpr int kStreamSweepCount =
    static_cast<int>(sizeof(kStreamSweep) / sizeof(kStreamSweep[0]));

}  // namespace

int RunAllModes(const char* mode, Pipeline* p, int chunks, int streams,
                int repeat) {
  if (std::strcmp(mode, "all") == 0) {
    ACL_CHECK(ConfigureChunks(p, 1));
    ACL_CHECK(MeasureSerial(p, "main", repeat, 5));
    ACL_CHECK(VerifyResult(p, "v1"));
    ACL_CHECK(ConfigureChunks(p, chunks));
    ACL_CHECK(MeasurePipelined(p, "main", "v2", 1, repeat, 5));
    ACL_CHECK(VerifyResult(p, "v2"));
    ACL_CHECK(MeasurePipelined(p, "main", "v3", 2, repeat, 5));
    ACL_CHECK(VerifyResult(p, "v3"));
    ACL_CHECK(MeasurePipelined(p, "main", "v4", streams, repeat, 5));
    ACL_CHECK(VerifyResult(p, "v4"));
    return ACL_SUCCESS;
  }
  if (std::strcmp(mode, "streams") == 0) {
    ACL_CHECK(ConfigureChunks(p, 1));
    ACL_CHECK(MeasureSerial(p, "sweep_streams", repeat, 3));
    ACL_CHECK(ConfigureChunks(p, chunks));
    for (int i = 0; i < kStreamSweepCount; ++i) {
      ACL_CHECK(MeasurePipelined(p, "sweep_streams", "pipe", kStreamSweep[i],
                                 repeat, 3));
    }
    return VerifyResult(p, "sweep_streams");
  }
  std::fprintf(stderr, "[ERR] api=main code=- msg=unknown mode %s\n", mode);
  return ACL_ERROR_INVALID_PARAM;
}

int main(int argc, char** argv) {
  const char* mode = (argc > 1) ? argv[1] : "all";
  const int total_mb = (argc > 2) ? std::atoi(argv[2]) : 64;
  const int chunks = (argc > 3) ? std::atoi(argv[3]) : 16;
  const int streams = (argc > 4) ? std::atoi(argv[4]) : 4;
  const int repeat = (argc > 5) ? std::atoi(argv[5]) : 4;

  // Every argument is checked before any resource is acquired: streams and
  // chunks index fixed-size arrays, and a chunk count that does not divide
  // the element count would leave a tail that is never computed.
  if (total_mb <= 0 || chunks <= 0 || chunks > kMaxChunks || streams <= 0 ||
      streams > kMaxStreams || repeat <= 0) {
    std::fprintf(stderr,
                 "[ERR] api=main code=- msg=usage: <mode> <total_mb greater "
                 "than 0> <chunks 1..%d> <streams 1..%d> <repeat greater "
                 "than 0>\n",
                 kMaxChunks, kMaxStreams);
    return ACL_ERROR_INVALID_PARAM;
  }
  const int64_t total_elems = static_cast<int64_t>(total_mb) * 1024 * 1024 / 4;
  if (total_elems % chunks != 0) {
    std::fprintf(stderr,
                 "[ERR] api=main code=- msg=total_elems %lld is not divisible "
                 "by chunks %d; the tail would never be computed\n",
                 static_cast<long long>(total_elems), chunks);
    return ACL_ERROR_INVALID_PARAM;
  }

  std::printf("[INFO] acl_stream start mode=%s\n", mode);
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));
  aclrtContext context = nullptr;
  ACL_CHECK(aclrtCreateContext(&context, kDeviceId));

  Pipeline pipeline = {};
  ACL_CHECK(AllocPipeline(&pipeline, total_elems, kMaxStreams));
  std::printf("[CFG] total_mb=%d elems=%lld chunks=%d streams=%d repeat=%d\n",
              total_mb, static_cast<long long>(total_elems), chunks, streams,
              repeat);

  const int ret = RunAllModes(mode, &pipeline, chunks, streams, repeat);

  ACL_CHECK(FreePipeline(&pipeline));
  ACL_CHECK(aclrtDestroyContext(context));
  ACL_CHECK(aclrtResetDevice(kDeviceId));
  ACL_CHECK(aclFinalize());

  std::printf("[RESULT] %s\n", (ret == ACL_SUCCESS) ? "PASS" : "FAILED");
  std::printf("[INFO] acl_stream finished\n");
  return (ret == ACL_SUCCESS) ? 0 : ret;
}


## 8. 编译与运行

程序调用了内置算子，因此除 Runtime 库外还要链接算子公共数据类型库与 Math 类算子库，链接参数取自 §4 自检得到的 `ACL_LIBS`。


In [ ]:
import os, subprocess

SRC = "src_stream/acl_stream.cpp"
EXE = "src_stream/acl_stream"

# g++ [源文件] -I[头文件目录] [库目录] [库名] -o [可执行文件]
cmd = (
    ["g++", SRC, "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", EXE]
)
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)


五种模式各运行一次。模式之后的四个参数依次是总数据量（MB）、分块数、流数、计算重复次数；扫描模式会覆盖其中被扫描的那一个。

**参数的选择依据**：总量取 64 MB；基准分块数取 16，对应每块 4 MB——选它是因为每块 4 MB 足够大，单块传输仍落在传输效率的平稳段上——块切得太细时，每块都要付一次传输固定开销，收益会被抵消；基准流数取 4；计算重复次数取 4。


In [ ]:
import subprocess

TOTAL_MB, CHUNKS, STREAMS, REPEAT = 64, 16, 4, 4
MODES = ["all", "streams"]
out = {}

for mode in MODES:
    args = [str(TOTAL_MB), str(CHUNKS), str(STREAMS), str(REPEAT)]
    proc = subprocess.run(
        ["./" + EXE, mode] + args, capture_output=True, text=True, timeout=1800
    )
    out[mode] = proc.stdout
    n_perf = proc.stdout.count("[PERF]")
    flag = "✅" if proc.returncode == 0 else "❌ 返回码 %d" % proc.returncode
    print("%-8s %s  记录行 %2d 条" % (mode, flag, n_perf))
    if proc.returncode != 0:
        print(proc.stderr)

print()
print(out["all"])


## 9. 解析输出

两类记录行分别对应两件事：`[STAGE]` 是串行基线的三段分解，`[PERF]` 是每个配置的下发耗时与总耗时。

加速比一律以**同一 `tag`、同一计算重复次数下的串行基线**为分母：基线与被比较的对象必须在同一配置下测得，否则比较的是两件不同的事情。

下面打印的 $S_\infty$ 与 $k$ 取当前值时的模型预测值都按 **H2D、计算、D2H 三段各占一种资源**计算（§5.1）。两个传输方向是否真的能同时进行取决于本机硬件，因此这两个数只能当作参考值——§12 ③ 会讨论实测与它们的差距从何而来。

还要注意：两者都由**整块**串行基线的三段分解算出，而流水版本每块只有 $1/k$ 的规模，因此它们并不是严格意义上的上界，实测可以超过。本 Notebook 一律称它们为模型预测值，不称上界。


In [ ]:
import re

NUMERIC = {
    "chunks",
    "streams",
    "repeat",
    "submit_ms",
    "total_ms",
    "h2d_ms",
    "comp_ms",
    "d2h_ms",
    "serial_ms",
    "ms",
    "chunk_kb",
}


def parse(text, tag):
    # 把形如 [TAG] k=v k=v 的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if not line.startswith(tag + " "):
            continue
        row = {}
        for item in line.split()[1:]:
            key, value = item.split("=", 1)
            row[key] = float(value) if key in NUMERIC else value
        rows.append(row)
    return rows


perf, stage = [], []
for mode in MODES:
    perf += parse(out[mode], "[PERF]")
    stage += parse(out[mode], "[STAGE]")

# 以 (tag, repeat) 为键索引基线，供加速比计算使用
BASE = {(r["tag"], r["repeat"]): r for r in stage}
for row in perf:
    row["speedup"] = BASE[(row["tag"], row["repeat"])]["serial_ms"] / row["total_ms"]


def bound_inf(st):
    # 分块数趋于无穷时的加速比上界：三段之和除以最长的一段
    return st["serial_ms"] / max(st["h2d_ms"], st["comp_ms"], st["d2h_ms"])


def bound_k(st, chunks):
    # 有限分块数下的上界：第一块走完全部阶段，其后每块只多占一个节拍
    total = st["serial_ms"]
    t_max = max(st["h2d_ms"], st["comp_ms"], st["d2h_ms"])
    return chunks * total / (total + (chunks - 1) * t_max)


print("解析得到 [PERF] %d 条、[STAGE] %d 条" % (len(perf), len(stage)))
print()
main = BASE[("main", float(REPEAT))]
print("串行基线的三段分解（总量 %d MB，计算重复 %d 次）：" % (TOTAL_MB, REPEAT))
for name, key in (("H2D", "h2d_ms"), ("计算", "comp_ms"), ("D2H", "d2h_ms")):
    print(
        "  %-4s %8.3f ms  占比 %5.1f%%"
        % (name, main[key], 100 * main[key] / main["serial_ms"])
    )
print("  合计 %8.3f ms" % main["serial_ms"])
print()
print("三段假设下的 S∞ = 三段之和 / 最长段 = %.3f" % bound_inf(main))
print("k = %d 时的模型预测值  = %.3f" % (CHUNKS, bound_k(main, CHUNKS)))
print("两者都建立在 §5.2 那个待检验的三段假设上，且由整块串行基线算出，不是严格上界。")


## 10. 版本对比与阶段分解

下表把四个版本并排列出。`submit_ms` 是主机把全部任务下发完所用的时间，`total_ms` 是全部 Stream 同步完成后的总耗时。两者的比值直接说明主机是否跟得上：**在采用异步接口的 v2–v4 上，比值远小于 1 说明主机把任务喂完时设备还有大量工作没做完，这正是流水线能够生效的前提。** v1 用的是同步复制接口，它没有独立的下发阶段，这一列对它没有意义，因此不参与比较，表中印为 `-`。

最后一列按 §5.1 的三段模型算出（§9 已说明），因此称作模型预测而不称上界——它由整块串行基线导出，并非严格上界。单流的 v1 与 v2 按 §6 不可能产生重叠，这一列对它们同样没有意义，也印为 `-`。


In [ ]:
print(
    "%-4s %-22s %8s %8s %10s %9s %10s"
    % ("版本", "配置", "下发ms", "总耗时ms", "下发/总耗时", "加速比", "模型预测")
)
print("-" * 80)
DESC = {
    "v1": "整块 · 同步 · 1 流",
    "v2": "%d 块 · 异步 · 1 流" % CHUNKS,
    "v3": "%d 块 · 异步 · 2 流" % CHUNKS,
    "v4": "%d 块 · 异步 · %d 流" % (CHUNKS, STREAMS),
}
for row in [r for r in perf if r["tag"] == "main"]:
    ver = row["ver"]
    # v1 用同步复制接口，没有独立的下发阶段，程序为它印 submit_ms = -1
    submitted = row["submit_ms"] >= 0
    # 单流版本按 §6 不可能产生重叠，三段模型的预测值对它没有意义
    single = int(row["streams"]) == 1
    print(
        "%-4s %-22s %8s %8.3f %10s %9.3f %10s"
        % (
            ver,
            DESC[ver],
            ("%.3f" % row["submit_ms"]) if submitted else "-",
            row["total_ms"],
            ("%.2f" % (row["submit_ms"] / row["total_ms"])) if submitted else "-",
            row["speedup"],
            "-" if single else ("%.3f" % bound_k(main, int(row["chunks"]))),
        )
    )
print()
print("下发两列的 - ：v1 用同步复制接口，没有独立的下发阶段。")
print("模型预测列的 - ：该版本只有一条 Stream，按 §6 不可能重叠，预测值恒为 1。")


下图左侧把串行基线的三段画成堆叠条，右侧把四个版本的加速比与理论上界画在一起。**读图时先看左边**：三段之中最长的那一段决定了右边那条上界线的高度。


In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_H2D, C_COMP, C_D2H = "#3B6FE0", "#E07A3B", "#2E9E6B"
C_BAR, C_LIMIT = "#3B6FE0", "#9AA5B1"

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2), dpi=120)

ax = axes[0]
left = 0.0
for key, color, name in (
    ("h2d_ms", C_H2D, "H2D"),
    ("comp_ms", C_COMP, "Compute"),
    ("d2h_ms", C_D2H, "D2H"),
):
    ax.barh([0], [main[key]], left=[left], color=color, label=name, height=0.5)
    ax.text(
        left + main[key] / 2,
        0,
        "%.1f" % main[key],
        ha="center",
        va="center",
        fontsize=9,
        color="white",
    )
    left += main[key]
ax.set_yticks([])
ax.set_xlabel("Time (ms)")
ax.set_title("Lab 3: v1 stage decomposition")
ax.legend(frameon=False, ncol=3, fontsize=9, loc="upper center")
ax.set_ylim(-0.6, 0.9)

ax = axes[1]
rows = [r for r in perf if r["tag"] == "main"]
names = [r["ver"] for r in rows]
ax.bar(names, [r["speedup"] for r in rows], color=C_BAR, width=0.55)
for i, r in enumerate(rows):
    ax.text(i, r["speedup"] + 0.03, "%.2f" % r["speedup"], ha="center", fontsize=9)
ax.axhline(1.0, color="#555555", lw=1.0, ls="-")
ax.axhline(
    bound_inf(main),
    color=C_LIMIT,
    lw=1.4,
    ls="--",
    label="S_inf = %.2f" % bound_inf(main),
)
ax.axhline(
    bound_k(main, CHUNKS),
    color="#E07A3B",
    lw=1.4,
    ls=":",
    label="three-stage model at k=%d: %.2f" % (CHUNKS, bound_k(main, CHUNKS)),
)
ax.set_ylabel("Speedup over v1")
ax.set_title("Lab 3: speedup by version")
ax.legend(frameon=False, fontsize=9, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


## 11. 流数扫描

固定分块数与计算量，把 Stream 数从 1 扫到 8。按 §2.1 的第二条规则，流数增加到某个点之后硬件资源不再充足，加速比应当趋于平缓。

请特别注意 **1 流与 2 流之间的那一段**：它是本实验的核心对照，量化的正是**同一条 Stream 内保序**这条规则所带来的损失。

读表时还要问一个问题：**曲线在第几条 Stream 上就已经不再上升了？** 这个位置在不同机器上并不相同，两种走向都会遇到：若在很小的流数上就完全持平，说明限制来自更靠下的某种资源而不是队列的条数，继续加流无效；若一路上升到扫描区间的末端，说明可并行的资源不止两组，本机的分组与 §5.2 的假设不同。请记录本机的位置，§12 ② 与 ③ 用的就是这条曲线。


In [ ]:
rows = sorted(
    [r for r in perf if r["tag"] == "sweep_streams" and r["ver"] == "pipe"],
    key=lambda r: r["streams"],
)
base_sw = BASE[("sweep_streams", float(REPEAT))]

print("%8s %10s %10s %10s %12s" % ("流数", "下发ms", "总耗时ms", "加速比", "占模型预测"))
print("-" * 52)
limit = bound_k(base_sw, CHUNKS)
for r in rows:
    print(
        "%8d %10.3f %10.3f %10.3f %11.0f%%"
        % (
            r["streams"],
            r["submit_ms"],
            r["total_ms"],
            r["speedup"],
            100 * r["speedup"] / limit,
        )
    )
print()
print("k = %d 时的三段模型预测值 = %.3f（§5.2 的假设尚待 §12 ② 判定）" % (CHUNKS, limit))


把上表画成曲线。横轴是流数，两条水平参考线分别是串行基线与 $k$ 取当前值时的三段模型预测值。**曲线与预测线之间的距离，把流数还不够与三段不均衡两种原因混在了一起，本节还不能把它们分开。** 若曲线越过了这条线，说明的不是流水线超速，而是这个预测值的前提或基线的取法存在问题。


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=120)
xs = [r["streams"] for r in rows]
ax.plot(
    xs,
    [r["speedup"] for r in rows],
    marker="o",
    ms=5,
    lw=1.8,
    color=C_BAR,
    label="measured",
)
ax.axhline(1.0, color="#555555", lw=1.0, label="v1 baseline")
ax.axhline(
    limit, color=C_LIMIT, lw=1.4, ls="--", label="three-stage model at k=%d: %.2f" % (CHUNKS, limit)
)
ax.set_xlabel("Number of streams")
ax.set_ylabel("Speedup over v1")
ax.set_title("Lab 3: speedup vs stream count (k=%d, R=%d)" % (CHUNKS, REPEAT))
ax.set_xticks(xs)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=9, loc="lower right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


## 12. 结果分析

> 以下结论针对**趋势规律**。本节不引用任何一次运行的具体数值，所有读数请以本机 §10 与 §11 的输出为准。

**① 分块与异步接口本身不产生重叠，重叠要等到第二条 Stream。**

按 §6 的预期，v2 在单条 Stream 上分块并改用异步接口，不应产生任何重叠，因此应当与 v1 基本持平。请对照本机 §10 的表：v2 的加速比是否明显高于 1？

无论高多少，判断它是否来自重叠只需要看 §11 中 1 流与 2 流之间的那一步：**若 1 流那一行与 v2 接近，而 1 流到 2 流之间有明显跳变，就说明真正的重叠是随第二条队列出现的，与分块本身无关。**

**这是本实验最核心的一条结论**：异步接口只是让主机不必等待，而**让不同硬件单元同时工作的前提是多条队列**。

**② 加速比随流数上升，但会在某一条上到顶。**

请看 §11 的曲线：加速比从第 1 条到第 2 条有一次明显跳变（这是同一条 Stream 内保序这条规则被打破的那一步），此后是很快持平，还是继续上升？请记录曲线在第几条 Stream 上不再上升。

到顶的原因是**某一类硬件资源被占满了**：§2.1 第二条规则说得很清楚，硬件资源充足时不同 Stream 上的任务并行执行，资源不足时它们仍然串行。再多的 Stream 只是多了几条空队列，不会变出新的硬件。

**两种走向在真实硬件上都会出现**，取决于本机可并行的资源有几组：

- 若曲线在**第 2 条之后很快持平**，说明可并行的资源只有两组——传输是一组、计算是一组，两个传输方向共用同一条链路、不能重叠；
- 若曲线在第 2 条之后**继续明显上升**，说明可并行的资源不止两组，两个传输方向至少部分可以重叠。

**③ 实测加速比达不到上界是正常的。**

§10 的表里每一行都给出了实测加速比与上界 $S_\infty$。两者之间通常有明显的差距，原因有三类，按可排查的顺序列出：

1. **流水线的填充开销尚未被分摊。** $S_\infty$ 是 $k \to \infty$ 的极限，而本实验的 $k$ 是有限的：第一块数据要完整经过三个阶段，这部分开销要由全部 $k$ 块共同分摊。$k$ 越小，实测值离极限越远。
2. **三段并非各占一组资源。** $S_\infty$ 的公式默认三段互不相干，而两个传输方向可能共用同一条链路——这正是 ② 要判定的事。若 ② 判定本机只有两组资源，真正的上界要低于 $S_\infty$。
3. **流数还不够。** 若 ② 的曲线在扫描区间末端仍在上升，说明资源还没占满，加速比还有余量。

---

### 🎓 结论

本实验完成了第七章第一个真正意义上的并行：**把串行的工作组织成流水线，收益来自让不同的硬件单元同时工作，而不是让任何一个单元变快。** 两条结论支撑它：其一，**重叠的前提是多条队列**——分块与异步接口本身都不产生重叠，1 流那一行把这一点单独隔离了出来；其二，**加速比有上界，上界由最慢的那一段决定**，而实测能不能接近它，取决于本机可并行的资源有几组。


## 13. 🔧 动手练习

> **提示**：本题要改 C++ 源码。修改源码后需要重新执行 `%%writefile` 单元格；由于 §7.1 为覆盖写、其后五个为追加写，**须从 7.1 开始按顺序重新执行**。

**直接判定两个传输方向能不能重叠。**

§12 ② 是由流数曲线的走向间接判定的，这个结论可以用一次专门的测量直接确认。

复制一份 `RunPipelined`，把其中的计算循环整段删掉，只保留 H2D 与 D2H 两次异步复制；为它加一个新的模式，分别用 1 条与 2 条 Stream 各跑一次。**这一版不要调用 `VerifyResult`**——输出缓冲区没有被计算写过，校验必然失败，这是预期的，本题只看耗时。

由 §9 打印的 $t_{\text{h2d}}$ 与 $t_{\text{d2h}}$ 先算出两个互斥的预测值：若两个方向能够重叠，2 条 Stream 的耗时应当接近 $\max(t_{\text{h2d}},\, t_{\text{d2h}})$；若不能重叠，应当接近 $t_{\text{h2d}} + t_{\text{d2h}}$。


## 14. 🤔 思考题

1. 本实验用 `aclrtSynchronizeStream` 逐条同步全部 Stream。若改为只同步最后一条，测出的 `total_ms` 会怎样变化？这个数字还有意义吗？

2. §2.2 指出默认 Stream 不与显式 Stream 隐式同步。设想一段代码：在显式 Stream 上下发计算，随后在默认 Stream 上下发 D2H 拷贝并同步默认 Stream，然后读取主机缓冲区。这段代码在功能测试中很可能通过。请说明它为什么是错的，以及在什么条件下会暴露出来。


## 15. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| Stream 的定义 | 在 Host 下发、在 Device 上执行的**任务队列**；同队列内保序 |
| 流间并行的前提 | **硬件资源充足时**才并行；资源不足时不同 Stream 上的任务仍然串行 |
| 默认 Stream | `aclrtSetDevice` / `aclrtCreateContext` 各隐式创建一个；传 `nullptr` 使用；**不与显式 Stream 隐式同步** |
| 默认 Stream 的其它规则 | 每 Context 一个，同 Context 的多线程共享；不能显式销毁；无 Stream 入参的接口不使用它 |
| Stream 数量上限 | 存在上限且含默认与内部 Stream；`aclrtGetStreamAvailableNum` 查剩余可用数 |
| 三级同步粒度 | `aclrtSynchronizeDevice` 当前 Context 下全部 Stream、`aclrtSynchronizeStream` 单条、`aclrtStreamQuery` **不阻塞** |
| **重叠的前提** | **多条队列**，而不是分块或异步接口本身；v2 与 v3 的对照即为此而设 |
| 流水线公式 | $T_{\text{pipe}} = \sum_j t_j + (k-1)\, t_{\max}$；教科书写法 $(k+s-1)\,t_{\max}$ 是各段相等时的特例 |
| 加速比上界 | $S_\infty = \left(\sum_j t_j\right) / t_{\max}$；只有三段完全相等时才等于 3 |
| 上界为什么达不到 | 三类原因：填充开销未被分摊（$k$ 有限）、两个传输方向可能共用一条链路、流数还不够 |
| 模型预测的前提 | 由整块串行基线的三段分解算出，而流水版本每块规模不同，**因此不是严格上界，实测可以超过它** |
| 测量规则 | 全部 Stream 都要同步；张量与内存在计时之外准备；分别记录下发与总耗时 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 的定义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">在 Host 下发、在 Device 上执行的<strong>任务队列</strong>；同队列内保序</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">流间并行的前提</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>硬件资源充足时</strong>才并行；资源不足时不同 Stream 上的任务仍然串行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSetDevice</code> / <code>aclrtCreateContext</code> 各隐式创建一个；传 <code>nullptr</code> 使用；<strong>不与显式 Stream 隐式同步</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">默认 Stream 的其它规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每 Context 一个，同 Context 的多线程共享；不能显式销毁；无 Stream 入参的接口不使用它</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Stream 数量上限</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">存在上限且含默认与内部 Stream；<code>aclrtGetStreamAvailableNum</code> 查剩余可用数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三级同步粒度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeDevice</code> 当前 Context 下全部 Stream、<code>aclrtSynchronizeStream</code> 单条、<code>aclrtStreamQuery</code> <strong>不阻塞</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>重叠的前提</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多条队列</strong>，而不是分块或异步接口本身；v2 与 v3 的对照即为此而设</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">流水线公式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$T_{\text{pipe}} = \sum_j t_j + (k-1)\, t_{\max}$；教科书写法 $(k+s-1)\,t_{\max}$ 是各段相等时的特例</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">加速比上界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S_\infty = \left(\sum_j t_j\right) / t_{\max}$；只有三段完全相等时才等于 3</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">上界为什么达不到</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三类原因：填充开销未被分摊（$k$ 有限）、两个传输方向可能共用一条链路、流数还不够</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型预测的前提</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由整块串行基线的三段分解算出，而流水版本每块规模不同，<strong>因此不是严格上界，实测可以超过它</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">测量规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全部 Stream 都要同步；张量与内存在计时之外准备；分别记录下发与总耗时</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **异构应用的性能问题，首先是数据搬运问题，其次才是计算问题。**

这条原则先落在单次搬运的代价上，本实验把它推进了一步：**搬运的代价不一定要被消除，也可以被藏起来。** 藏的手段是让搬运与计算发生在不同的硬件单元上并同时进行，而能藏多少，取决于**被藏起来的那一段与掩盖它的那一段是否匹配**——能藏起来的至多是较短的那一段。四个版本中没有任何一处改动涉及被搬运的数据本身或计算的内容，全部性能差异都来自任务的组织方式。

### 本实验没有回答的问题

本实验全部使用主机侧墙钟时间，用下发一段、同步、停计时的方式测三段分解。这个方法把主机侧的下发与同步开销一并计入设备侧的耗时，因此**主机侧测到的计算耗时里究竟包含了多少下发开销，本实验分不出来**——要分开，需要设备侧的计时手段。

另一件本实验刻意回避的事是跨流依赖。本实验的每一块数据都在同一条 Stream 上走完三个阶段，块与块之间没有依赖，因此不需要任何跨流同步机制。**若把三段拆到不同 Stream 上**，第 $i$ 块的计算就必须等它自己的 H2D 完成，而两者在不同的队列上——这层依赖用同步接口表达代价过高，需要更细粒度的机制。